# Import

In [ ]:
!pip install torch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Input

In [ ]:
# Install dependencies as needed:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "jena_climate_2009_2016.csv"

# Load the latest version
dataset = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "stytch16/jena-climate-2009-2016",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:")
display(dataset.head())

/tmp/ipykernel_1526/3593041896.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  dataset = kagglehub.load_dataset(


Using Colab cache for faster access to the 'jena-climate-2009-2016' dataset.
First 5 records:


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [ ]:
jena_climate_dataset = dataset.copy()

# Clean and Scale the Data

In [ ]:
# 1. Load and downsample
jena_climate_dataset = jena_climate_dataset.iloc[5::6].copy()
jena_climate_dataset = jena_climate_dataset.drop('Date Time', axis=1)

# 2. Separate the Features (X) and the Target (y)
target_col = 'T (degC)'

# Note: We use double brackets [[ ]] for y to keep it a 2D array, which the scaler requires
y_raw = jena_climate_dataset[[target_col]].values
X_raw = jena_climate_dataset.drop(target_col, axis=1).values

# 3. Create TWO separate scalers
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_scaled = feature_scaler.fit_transform(X_raw)
y_scaled = target_scaler.fit_transform(y_raw)

## data preparation for Pytorch

In [ ]:
# --- Updated Dataset Class ---
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, lookback_steps, forecast_horizon=1):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.lookback = lookback_steps
        self.horizon = forecast_horizon

    def __len__(self):
        return len(self.X) - self.lookback - self.horizon + 1

    def __getitem__(self, index):
        # Window of past features: [Sequence Length, Features]
        x_window = self.X[index : index + self.lookback]

        # Target value in the future: [1]
        target_index = index + self.lookback + self.horizon - 1
        y_target = self.y[target_index]

        return x_window, y_target

In [ ]:
# --- 1. Calculate the Split Indices ---
# We will use a standard chronological split: 80% Train, 10% Validation, 10% Test
total_samples = len(X_scaled)
train_end = int(total_samples * 0.8)
val_end = int(total_samples * 0.9)

# --- 2. Chronologically Slice the Arrays ---
X_train = X_scaled[:train_end]
y_train = y_scaled[:train_end]

X_val = X_scaled[train_end:val_end]
y_val = y_scaled[train_end:val_end]

X_test = X_scaled[val_end:]
y_test = y_scaled[val_end:]

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Testing samples: {len(X_test)}")

# --- 3. Instantiate the Three Datasets ---
lookback = 24

train_dataset = TimeSeriesDataset(X_train, y_train, lookback_steps=lookback)
val_dataset   = TimeSeriesDataset(X_val, y_val, lookback_steps=lookback)
test_dataset  = TimeSeriesDataset(X_test, y_test, lookback_steps=lookback)

# --- 4. Create the DataLoaders ---
# NOTE: It is safe (and often beneficial) to shuffle the sliding windows
# in the TRAINING loader only, to break up epoch-to-epoch correlations.
# NEVER shuffle the validation or test loaders.
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

Training samples: 56072
Validation samples: 7009
Testing samples: 7010


In [ ]:
# Grab one batch of data
for X_batch, y_batch in train_dataloader:
    print(f"Input X shape:  {X_batch.shape}")
    print(f"Target y shape: {y_batch.shape}")
    break

Input X shape:  torch.Size([32, 24, 13])
Target y shape: torch.Size([32, 1])


# Deep Learning models:

## DNN

### architecture

In [ ]:
class BaselineDNN(nn.Module):
    def __init__(self, lookback, features):
        super().__init__()
        # 1. The flattener
        self.flatten = nn.Flatten()

        # 2. The network stack
        # Input dimension is 24 * 13 = 312
        self.net = nn.Sequential(
            nn.Linear(lookback * features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Output a single prediction
        )

    def forward(self, x):
        # Input 'x' shape: [32, 24, 13]

        x = self.flatten(x)
        # Shape is now: [32, 312]

        predictions = self.net(x)
        # Shape is now: [32, 1]

        return predictions

### training & Dianogsis

In [ ]:
# 1. Hyperparameters & Model Setup
epochs = 15
learning_rate = 0.001

# Instantiate the model with your lookback (24) and features (13)
model = BaselineDNN(lookback=24, features=13)

# Loss function and Optimizer
criterion = nn.MSELoss()  # MSE is perfect for regression
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Starting Training Loop...\n")

# 2. The Core Training & Validation Loop
for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()  # Set model to training mode
    train_loss = 0.0

    for X_batch, y_batch in train_dataloader:
        # X_batch shape: [32, 24, 13], y_batch shape: [32, 1]

        # Forward pass: Compute predicted y by passing x to the model
        predictions = model(X_batch)

        # Compute loss
        loss = criterion(predictions, y_batch)

        # Zero gradients, perform a backward pass, and update the weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        train_loss += loss.item() * X_batch.size(0)

    # Calculate average training loss for this epoch
    epoch_train_loss = train_loss / len(train_dataloader.dataset)

    # --- VALIDATION PHASE ---
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0

    with torch.no_grad():  # Turn off gradient computation for validation
        for X_batch, y_batch in val_dataloader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)

    epoch_val_loss = val_loss / len(val_dataloader.dataset)

    # Print metrics every epoch
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss (MSE): {epoch_train_loss:.5f} | Val Loss (MSE): {epoch_val_loss:.5f}")


# 3.  (Evaluation on Unseen Data)
print("\nTraining Complete. Evaluating on Test Set...")
model.eval()

all_predictions = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        predictions = model(X_batch)

        # Collect predictions and actuals as numpy arrays
        all_predictions.append(predictions.numpy())
        all_actuals.append(y_batch.numpy())

# Stack all batches into single arrays
all_predictions = np.vstack(all_predictions)
all_actuals = np.vstack(all_actuals)

# 4. Post-Processing: Convert Scaled Z-Scores back to Degrees Celsius
# This uses the target_scaler we created during the data cleaning step
true_celsius = target_scaler.inverse_transform(all_actuals)
pred_celsius = target_scaler.inverse_transform(all_predictions)

# 5. Calculate Final Real-World Accuracy Metrics
final_mae = mean_absolute_error(true_celsius, pred_celsius)
final_mse = mean_squared_error(true_celsius, pred_celsius)
final_rmse = np.sqrt(final_mse)

print("\n=========================================")
print("          FINAL TEST RESULTS             ")
print("=========================================")
print(f"Mean Absolute Error (MAE):    {final_mae:.4f} °C")
print(f"Root Mean Squared Error (RMSE): {final_rmse:.4f} °C")
print("=========================================")

Starting Training Loop...

Epoch 01/15 | Train Loss (MSE): 0.01849 | Val Loss (MSE): 1.08927
Epoch 02/15 | Train Loss (MSE): 0.01017 | Val Loss (MSE): 1.78857
Epoch 03/15 | Train Loss (MSE): 0.00920 | Val Loss (MSE): 2.05490
Epoch 04/15 | Train Loss (MSE): 0.00885 | Val Loss (MSE): 1.99205
Epoch 05/15 | Train Loss (MSE): 0.00855 | Val Loss (MSE): 2.12955
Epoch 06/15 | Train Loss (MSE): 0.00824 | Val Loss (MSE): 1.68355
Epoch 07/15 | Train Loss (MSE): 0.00819 | Val Loss (MSE): 1.98798
Epoch 08/15 | Train Loss (MSE): 0.00795 | Val Loss (MSE): 1.79119
Epoch 09/15 | Train Loss (MSE): 0.00788 | Val Loss (MSE): 2.27582
Epoch 10/15 | Train Loss (MSE): 0.00774 | Val Loss (MSE): 1.60017
Epoch 11/15 | Train Loss (MSE): 0.00769 | Val Loss (MSE): 1.47014
Epoch 12/15 | Train Loss (MSE): 0.00756 | Val Loss (MSE): 1.71140
Epoch 13/15 | Train Loss (MSE): 0.00748 | Val Loss (MSE): 1.66746
Epoch 14/15 | Train Loss (MSE): 0.00738 | Val Loss (MSE): 1.98619
Epoch 15/15 | Train Loss (MSE): 0.00734 | Val Los

## LSTM

### architecture

In [ ]:
# ==========================================
# 1. THE LSTM ARCHITECTURE
# ==========================================
class ForecastingLSTM(nn.Module):
    def __init__(self, input_features, hidden_dim):
        super().__init__()

        # batch_first=True expects input shape: [Batch Size, Sequence Length, Features]
        self.lstm = nn.LSTM(
            input_size=input_features,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        # Maps the final hidden memory state to a single continuous prediction
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: [32, 24, 13]
        lstm_out, (hidden_state, cell_state) = self.lstm(x)

        # Slice the tensor to grab the memory state of the very last hour in the sequence
        # lstm_out[:, -1, :] grabs the 24th time step for every item in the batch
        last_time_step = lstm_out[:, -1, :]

        predictions = self.linear(last_time_step)
        # predictions shape: [32, 1]

        return predictions

### training & Dianogsis

In [ ]:
# ==========================================
# 2. HYPERPARAMETERS & SETUP
# ==========================================
epochs = 15
learning_rate = 0.0005 # LSTMs often benefit from a slightly lower LR than standard DNNs
weight_decay = 1e-5    # L2 regularization to help prevent overfitting

# Instantiate the model (13 features in, 64 hidden memory cells)
model = ForecastingLSTM(input_features=13, hidden_dim=64)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print("Starting LSTM Training Loop...\n")

# ==========================================
# 3. THE TRAINING & VALIDATION LOOP
# ==========================================
for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_dataloader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = train_loss / len(train_dataloader.dataset)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_dataloader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)

    epoch_val_loss = val_loss / len(val_dataloader.dataset)

    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss (MSE): {epoch_train_loss:.5f} | Val Loss (MSE): {epoch_val_loss:.5f}")


# ==========================================
# 4. THE TESTING PHASE (UNSEEN DATA)
# ==========================================
print("\nTraining Complete. Evaluating LSTM on Test Set...")
model.eval()

all_predictions = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        predictions = model(X_batch)

        all_predictions.append(predictions.numpy())
        all_actuals.append(y_batch.numpy())

all_predictions = np.vstack(all_predictions)
all_actuals = np.vstack(all_actuals)

# Convert Scaled Z-Scores back to Degrees Celsius
true_celsius = target_scaler.inverse_transform(all_actuals)
pred_celsius = target_scaler.inverse_transform(all_predictions)

# Calculate Final Metrics
final_mae = mean_absolute_error(true_celsius, pred_celsius)
final_mse = mean_squared_error(true_celsius, pred_celsius)
final_rmse = np.sqrt(final_mse)

print("\n=========================================")
print("         FINAL LSTM TEST RESULTS         ")
print("=========================================")
print(f"Mean Absolute Error (MAE):    {final_mae:.4f} °C")
print(f"Root Mean Squared Error (RMSE): {final_rmse:.4f} °C")
print("=========================================")

Starting LSTM Training Loop...

Epoch 01/15 | Train Loss (MSE): 0.02882 | Val Loss (MSE): 0.00912
Epoch 02/15 | Train Loss (MSE): 0.00791 | Val Loss (MSE): 0.00782
Epoch 03/15 | Train Loss (MSE): 0.00740 | Val Loss (MSE): 0.00789
Epoch 04/15 | Train Loss (MSE): 0.00722 | Val Loss (MSE): 0.00835
Epoch 05/15 | Train Loss (MSE): 0.00708 | Val Loss (MSE): 0.00903
Epoch 06/15 | Train Loss (MSE): 0.00704 | Val Loss (MSE): 0.00852
Epoch 07/15 | Train Loss (MSE): 0.00698 | Val Loss (MSE): 0.00783
Epoch 08/15 | Train Loss (MSE): 0.00692 | Val Loss (MSE): 0.00840
Epoch 09/15 | Train Loss (MSE): 0.00691 | Val Loss (MSE): 0.00925
Epoch 10/15 | Train Loss (MSE): 0.00689 | Val Loss (MSE): 0.00791
Epoch 11/15 | Train Loss (MSE): 0.00684 | Val Loss (MSE): 0.00837
Epoch 12/15 | Train Loss (MSE): 0.00681 | Val Loss (MSE): 0.00845
Epoch 13/15 | Train Loss (MSE): 0.00681 | Val Loss (MSE): 0.00745
Epoch 14/15 | Train Loss (MSE): 0.00681 | Val Loss (MSE): 0.00775
Epoch 15/15 | Train Loss (MSE): 0.00676 | Va

## RNN

### architecture

In [ ]:
import torch.nn as nn

class ForecastingRNN(nn.Module):
    def __init__(self, input_features, hidden_dim):
        super().__init__()

        # We swap nn.LSTM for nn.RNN
        self.rnn = nn.RNN(
            input_size=input_features,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # IMPORTANT DIFFERENCE: nn.RNN only outputs the sequence and a single hidden state
        # It does NOT output a (hidden_state, cell_state) tuple like an LSTM does.
        rnn_out, hidden_state = self.rnn(x)

        # We still slice the tensor to grab the 24th time step (the final hour)
        last_time_step = rnn_out[:, -1, :]

        predictions = self.linear(last_time_step)

        return predictions

### training & Dianogsis

In [ ]:
# ==========================================
# 2. HYPERPARAMETERS & SETUP
# ==========================================
epochs = 15
learning_rate = 0.0005
weight_decay = 1e-5

# Swap to the Vanilla RNN architecture
model = ForecastingRNN(input_features=13, hidden_dim=64)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print("Starting RNN Training Loop...\n")

# ==========================================
# 3. THE TRAINING & VALIDATION LOOP
# ==========================================
for epoch in range(epochs):

    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_dataloader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = train_loss / len(train_dataloader.dataset)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_dataloader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item() * X_batch.size(0)

    epoch_val_loss = val_loss / len(val_dataloader.dataset)

    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss (MSE): {epoch_train_loss:.5f} | Val Loss (MSE): {epoch_val_loss:.5f}")


# ==========================================
# 4. THE TESTING PHASE (UNSEEN DATA)
# ==========================================
print("\nTraining Complete. Evaluating RNN on Test Set...")
model.eval()

all_predictions = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        predictions = model(X_batch)

        all_predictions.append(predictions.numpy())
        all_actuals.append(y_batch.numpy())

all_predictions = np.vstack(all_predictions)
all_actuals = np.vstack(all_actuals)

# Convert Scaled Z-Scores back to Degrees Celsius
true_celsius = target_scaler.inverse_transform(all_actuals)
pred_celsius = target_scaler.inverse_transform(all_predictions)

# Calculate Final Metrics
final_mae = mean_absolute_error(true_celsius, pred_celsius)
final_mse = mean_squared_error(true_celsius, pred_celsius)
final_rmse = np.sqrt(final_mse)

print("\n=========================================")
print("         FINAL RNN TEST RESULTS          ")
print("=========================================")
print(f"Mean Absolute Error (MAE):    {final_mae:.4f} °C")
print(f"Root Mean Squared Error (RMSE): {final_rmse:.4f} °C")
print("=========================================")

Starting RNN Training Loop...

Epoch 01/15 | Train Loss (MSE): 0.01752 | Val Loss (MSE): 0.00953
Epoch 02/15 | Train Loss (MSE): 0.00875 | Val Loss (MSE): 0.00904
Epoch 03/15 | Train Loss (MSE): 0.00796 | Val Loss (MSE): 0.00895
Epoch 04/15 | Train Loss (MSE): 0.00768 | Val Loss (MSE): 0.00880
Epoch 05/15 | Train Loss (MSE): 0.00761 | Val Loss (MSE): 0.00799
Epoch 06/15 | Train Loss (MSE): 0.00749 | Val Loss (MSE): 0.00774
Epoch 07/15 | Train Loss (MSE): 0.00745 | Val Loss (MSE): 0.00863
Epoch 08/15 | Train Loss (MSE): 0.00741 | Val Loss (MSE): 0.00791
Epoch 09/15 | Train Loss (MSE): 0.00737 | Val Loss (MSE): 0.00767
Epoch 10/15 | Train Loss (MSE): 0.00734 | Val Loss (MSE): 0.00773
Epoch 11/15 | Train Loss (MSE): 0.00732 | Val Loss (MSE): 0.00897
Epoch 12/15 | Train Loss (MSE): 0.00731 | Val Loss (MSE): 0.00826
Epoch 13/15 | Train Loss (MSE): 0.00728 | Val Loss (MSE): 0.00814
Epoch 14/15 | Train Loss (MSE): 0.00726 | Val Loss (MSE): 0.00797
Epoch 15/15 | Train Loss (MSE): 0.00719 | Val